# 02 — Data Cleaning, Solid Mask Generation, Splitting & Preprocessing



- Duplicate auditing & removal, hollow-to-solid binary mask generation (with border-clipped ellipse repair), and leakage-free exam-grouped HC-stratified **70% / 15% / 15%** splitting.
- Dimension standardization (+ `mm/px` recalibration), intensity normalization, and edge-preserving speckle-reduction filtering.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Image, display

from src.config import (
    RAW_TRAIN_DIR,
    RAW_TRAIN_CSV,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    SPLITS_DIR,
    FIGURES_DIR,
    COL_FILENAME,
    COL_PIXEL_SIZE,
    COL_HC_MM,
    RANDOM_SEED,
    DEFAULT_TARGET_SIZE,
)
from src.data_prep.cleaning import clean_hc18_metadata
from src.data_prep.masks import generate_solid_masks
from src.data_prep.splitting import create_grouped_stratified_splits
from src.data_prep.run_data_prep import plot_mask_verification, plot_split_distributions

print("Raw train dir:", RAW_TRAIN_DIR)
print("Raw train CSV:", RAW_TRAIN_CSV)

## 1. Duplicate Audit & Resolution 
Audits both raw-file SHA-256 and decoded pixel-array SHA-256 across all 999 ultrasound images and annotations, reconstructs multi-plane `exam_group_id` clusters, drops redundant duplicates, and exports `data/interim/duplicate_audit.csv` and `data/interim/cleaned_metadata.csv`.

In [ ]:
cleaned_df, duplicate_audit_df = clean_hc18_metadata()
print(f"Cleaned samples: {len(cleaned_df)} (from 999) across {cleaned_df['exam_group_id'].nunique()} unique exam groups.")
display(duplicate_audit_df)

## 2. Hollow Ellipse Contour to Solid Binary Mask Conversion 
Flood-fills all 994 cleaned `*_Annotation.png` contours into solid binary masks (`0` = background, `255` = head). Automatically detects the 4 border-clipped annotations (`231_HC`, `427_2HC`, `776_HC`, `977_HC`) whose ellipses touch the image boundary and repairs them via exact ellipse fitting (`cv2.fitEllipse`).

In [ ]:
cleaned_df = generate_solid_masks(cleaned_df)
print("Mask fill methods:", cleaned_df["mask_fill_method"].value_counts().to_dict())
mean_fg = cleaned_df["fg_ratio"].mean()
print(f"Foreground pixel share: {mean_fg*100:.2f}% | Background: {(1-mean_fg)*100:.2f}% | Ratio bg/fg: {(1-mean_fg)/mean_fg:.3f}")

mask_fig_path = plot_mask_verification(cleaned_df)
display(Image(filename=str(mask_fig_path)))

## 3. Leakage-Free Exam-Grouped & HC-Stratified Split 
Splits the 994 cleaned images strictly by `exam_group_id` (779 exams) into **Train (70%) / Validation (15%) / Internal Test (15%)**, stratified across HC size strata (`<100`, `100-150`, `150-200`, `200-250`, `250-300`, `>300` mm), with automated assertions guaranteeing zero overlap in exams, filenames, and pixel hashes.

In [ ]:
splits, split_summary_df = create_grouped_stratified_splits(cleaned_df)
display(split_summary_df)

split_fig_path = plot_split_distributions(splits)
display(Image(filename=str(split_fig_path)))

## 4. Resizing, Pixel-Size Recalibration, Normalization & Edge-Preserving Despeckling
- Standardize images and binary masks to `DEFAULT_TARGET_SIZE` (`H x W`), recording `pixel_size_x_mm` and `pixel_size_y_mm`.
- Normalize grayscale intensities to `[0, 1]`.
- Apply edge-preserving speckle-reduction filtering (e.g., Bilateral, Anisotropic Diffusion, Non-Local Means) and save both raw-normalized and despeckled variants to `data/processed/`.

In [ ]:
# TODO (Member 2): Implement src/preprocessing/ pipeline and export to PROCESSED_DATA_DIR